# 🕵️‍♂️ Exp B: Deep Variational Autoencoder (`PyTorch` VAE) + Graph Network Ring Detection vs. `IsolationForest`
**Author:** Akmal Khan, PhD  
**Repository:** [github.com/akmalkhaniub/enterprise-banking-ds-platform](https://github.com/akmalkhaniub/enterprise-banking-ds-platform)

### 🎯 Research Question
Fraudsters in Banking (mule structuring rings), E-Commerce (organized refund/voucher syndicates), and Telecom (SIM-box bypass) operate in **cyclic transfer graphs**. Can a **PyTorch Deep Autoencoder / VAE** combined with **Graph PageRank & 2-Hop Cycle Centrality** detect collusive fraud rings that standard tabular `IsolationForest` misses?

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 1. Simulate a 15,000-Transaction Directed Network across 800 Accounts with Injected Collusive Mule Rings
n_accounts = 800
n_txns = 15000
ring_accounts = set(range(0, 45))  # 9 collusive 5-node fraud rings

edges = []
for i in range(n_txns):
    if i % 25 == 0:
        # Collusive ring transfer (subtle individual amount just below threshold, high cyclic velocity)
        ring_idx = (i // 25) % 9
        base_node = ring_idx * 5
        u = base_node + (i % 5)
        v = base_node + ((i + 1) % 5)
        amt = np.random.uniform(220_000, 248_000)
        hour = np.random.choice([1, 2, 3, 23])
        is_fraud = 1
    else:
        u = np.random.randint(45, n_accounts)
        v = np.random.randint(45, n_accounts)
        if u == v:
            v = (v + 1) % n_accounts
        amt = np.random.lognormal(mean=10.2, sigma=0.75)
        hour = np.random.choice(np.arange(8, 21))
        is_fraud = 0
    edges.append({'src': u, 'dst': v, 'amount': amt, 'hour': hour, 'is_odd_hour': int(hour in [0,1,2,3,4,23]), 'is_fraud': is_fraud})

df_txns = pd.DataFrame(edges)
print(f'Transactions: {len(df_txns):,} | Fraud Ring Rate: {df_txns["is_fraud"].mean()*100:.2f}%')

In [ ]:
# 2. Graph Feature Engineering (NetworkX Directed Graph: PageRank, In/Out Degree Ratio, Reciprocal/Cycle Participation)
G = nx.DiGraph()
for row in df_txns.itertuples():
    if G.has_edge(row.src, row.dst):
        G[row.src][row.dst]['weight'] += row.amount
        G[row.src][row.dst]['count'] += 1
    else:
        G.add_edge(row.src, row.dst, weight=row.amount, count=1)

pagerank = nx.pagerank(G, weight='weight')
in_deg = dict(G.in_degree(weight='count'))
out_deg = dict(G.out_degree(weight='count'))
clustering_coeff = nx.clustering(G.to_undirected())

df_txns['src_pagerank'] = df_txns['src'].map(pagerank).fillna(0)
df_txns['dst_pagerank'] = df_txns['dst'].map(pagerank).fillna(0)
df_txns['src_degree_balance'] = df_txns['src'].map(in_deg).fillna(0) / (df_txns['src'].map(out_deg).fillna(0) + 1.0)
df_txns['src_triangle_clustering'] = df_txns['src'].map(clustering_coeff).fillna(0)
df_txns['edge_tx_velocity'] = [G[u][v]['count'] for u, v in zip(df_txns['src'], df_txns['dst'])]

tabular_cols = ['amount', 'hour', 'is_odd_hour']
graph_augmented_cols = ['amount', 'hour', 'is_odd_hour', 'src_pagerank', 'dst_pagerank', 'src_degree_balance', 'src_triangle_clustering', 'edge_tx_velocity']

In [ ]:
# 3. Train PyTorch Deep Variational Autoencoder (VAE) Strictly on Normal Regime (Semi-Supervised Manifold Learning)
normal_mask = (df_txns['is_fraud'].values == 0)
scaler_tab = StandardScaler().fit(df_txns.loc[normal_mask, tabular_cols].values.astype(np.float32))
X_tab = scaler_tab.transform(df_txns[tabular_cols].values.astype(np.float32))

scaler_graph = StandardScaler().fit(df_txns.loc[normal_mask, graph_augmented_cols].values.astype(np.float32))
X_graph = scaler_graph.transform(df_txns[graph_augmented_cols].values.astype(np.float32))
X_graph_normal = X_graph[normal_mask]

class DeepAnomalyVAE(nn.Module):
    def __init__(self, in_dim, latent_dim=4):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, 32), nn.BatchNorm1d(32), nn.LeakyReLU(0.1), nn.Linear(32, 16), nn.LeakyReLU(0.1))
        self.mu = nn.Linear(16, latent_dim)
        self.logvar = nn.Linear(16, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 16), nn.LeakyReLU(0.1), nn.Linear(16, 32), nn.LeakyReLU(0.1), nn.Linear(32, in_dim))
    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std) if self.training else mu
        recon = self.dec(z)
        return recon, mu, logvar

vae = DeepAnomalyVAE(in_dim=X_graph.shape[1]).to(device)
opt = torch.optim.AdamW(vae.parameters(), lr=3e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(torch.tensor(X_graph_normal).to(device)), batch_size=256, shuffle=True)

vae.train()
for epoch in range(25):
    for (xb,) in loader:
        opt.zero_grad()
        recon, mu, logvar = vae(xb)
        recon_loss = ((xb - recon) ** 2).mean()
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        (recon_loss + 0.05 * kl_loss).backward()
        opt.step()

vae.eval()
with torch.no_grad():
    X_t = torch.tensor(X_graph).to(device)
    recon_t, mu_t, logvar_t = vae(X_t)
    vae_scores = ((X_t - recon_t) ** 2).mean(dim=1).cpu().numpy()

In [ ]:
# 4. Benchmark: Tabular IsolationForest vs Graph-IsolationForest vs PyTorch Deep Graph-VAE
iso_tab = IsolationForest(n_estimators=150, contamination=0.04, random_state=42).fit(X_tab)
iso_tab_scores = -iso_tab.score_samples(X_tab)

iso_graph = IsolationForest(n_estimators=150, contamination=0.04, random_state=42).fit(X_graph)
iso_graph_scores = -iso_graph.score_samples(X_graph)

ensemble_scores = 0.55 * (vae_scores / vae_scores.std()) + 0.45 * (iso_graph_scores / iso_graph_scores.std())
y_true = df_txns['is_fraud'].values

def score_model(name, scores):
    thresh = np.percentile(scores, 96.0)
    preds = (scores >= thresh).astype(int)
    return {
        'Model Architecture': name,
        'ROC-AUC': round(roc_auc_score(y_true, scores), 4),
        'PR-AUC': round(average_precision_score(y_true, scores), 4),
        'F1 @ Top 4%': round(f1_score(y_true, preds), 4)
    }

results_df = pd.DataFrame([
    score_model('1. Tabular IsolationForest (Amount + Hour Only)', iso_tab_scores),
    score_model('2. Graph-Augmented IsolationForest (+PageRank & Velocity)', iso_graph_scores),
    score_model('3. PyTorch Deep Graph-VAE (Reconstruction Error)', vae_scores),
    score_model('4. Hybrid Deep Graph-VAE + IsolationForest Ensemble', ensemble_scores),
])
print('=== COLLUSIVE FRAUD RING DETECTION LEADERBOARD ===')
print(results_df.to_string(index=False))